# Test des paramètres du modèle pour le déploiement sur la devboard 

In [1]:
import tensorflow as tf

interpreter = tf.lite.Interpreter(model_path="../final_model_deploy_int8.tflite")
interpreter.allocate_tensors()

inp = interpreter.get_input_details()[0]
out = interpreter.get_output_details()[0]

print("Entrée")
print("shape      :", inp["shape"])
print("dtype      :", inp["dtype"])
print("scale      :", inp["quantization"][0])
print("zero_point :", inp["quantization"][1])

print("\nSortie")
print("shape      :", out["shape"])
print("dtype      :", out["dtype"])
print("scale      :", out["quantization"][0])
print("zero_point :", out["quantization"][1])

I0000 00:00:1785227184.519544   60547 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785227185.035726   60547 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785227186.084698   60547 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


Entrée
shape      : [ 1 64 50  1]
dtype      : <class 'numpy.int8'>
scale      : 0.032686252146959305
zero_point : -32

Sortie
shape      : [1 3]
dtype      : <class 'numpy.int8'>
scale      : 0.00390625
zero_point : -128


/home/f24achba/Documents/KWS_model/52-Hertz-KWS/.venv_test/lib/python3.12/site-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


=> compatible avec ce qui est généré par le software !

# Test sur target 

## Génération du fichier .npy

In [2]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "/home/f24achba/Documents/KWS_model/52-Hertz-KWS/src")))

Le pipeline prend en entrée un audio brut pour tester toute la chaine audio, on prend des données test "talky" (20 fichiers des données d'entrainement)

In [3]:
import numpy as np
import tensorflow as tf
import glob

# Charge ton pipeline existant
from preprocessing import AudioPipeline
pipeline = AudioPipeline("/home/f24achba/Documents/KWS_model/52-Hertz-KWS/configuration/config.yaml")

# Charge l'interpréteur TFLite (modèle déjà quantifié int8)
interpreter = tf.lite.Interpreter(model_path="../final_model_deploy_int8.tflite")
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

in_scale, in_zero_point = input_details['quantization']
print(f"Quantization entrée : scale={in_scale}, zero_point={in_zero_point}")
print(f"Type d'entrée attendu : {input_details['dtype']}")
print(f"Type de sortie : {output_details['dtype']}")

# Liste de fichiers de validation (adapte le chemin/pattern à ton dataset)
val_files = glob.glob("/home/f24achba/Documents/data/données_test/*.wav", recursive=True)[:20]

X_val = []
Y_val = []

for i, path in enumerate(val_files):    
    audio = pipeline.load_audio(path)
    audio = pipeline.normalize_audio(audio)

    audio.astype(np.float32).tofile(f"test_talky_{i:02d}.bin") #Ajoute cette ligne pour sauvegarder l'audio normalisé en binaire
    
    log_mel = pipeline.extract_log_mel(audio)              # (64, 50) float32

    # Quantization identique à celle attendue par le modèle
    log_mel_q = np.round(log_mel / in_scale + in_zero_point).astype(np.int8)
    log_mel_q = log_mel_q[..., np.newaxis]                  # (64, 50, 1)

    # Inférence de référence via TFLite (sur PC, modèle original)
    interpreter.set_tensor(input_details['index'], log_mel_q[np.newaxis, ...])
    interpreter.invoke()
    out_ref = interpreter.get_tensor(output_details['index'])[0]  # (3,)

    X_val.append(log_mel_q)
    Y_val.append(out_ref)

X_val = np.stack(X_val)   # shape (N, 64, 50, 1), dtype int8
Y_val = np.stack(Y_val)   # shape (N, 3), dtype selon ta sortie (int8 ou float32)

np.save("../Tests_results/val_input_pipeline_C.npy", X_val)
np.save("../Tests_results/val_output_pipeline_C.npy", Y_val)
np.savetxt("../Tests_results/val_output_pipeline_C_ref.txt", Y_val, fmt="%.6f")

print(f"{len(X_val)} échantillons exportés")
print(f"X_val shape={X_val.shape}, dtype={X_val.dtype}")
print(f"Y_val shape={Y_val.shape}, dtype={Y_val.dtype}")

Quantization entrée : scale=0.032686252146959305, zero_point=-32
Type d'entrée attendu : <class 'numpy.int8'>
Type de sortie : <class 'numpy.int8'>
20 échantillons exportés
X_val shape=(20, 64, 50, 1), dtype=int8
Y_val shape=(20, 3), dtype=int8
